# 02 — Bronze Layer
**Purpose:** ingest raw Parquet into a Delta table. Add lineage metadata. No business transforms.   
**Output table:** `workspace.taxi.bronze_yellow_taxi`   
**Metadata columns added:** `ingestion_timestamp`, `source_file`

### Why metadata columns?

We don't transform the data here — Bronze must stay loyal to source so we can re-derive Silver if a downstream rule changes. We *do* tag every row with `ingestion_timestamp` and `source_file` because that lineage is impossible to recover later. The `try`/`except` wrapping ensures a write failure still produces an audit row with `status = FAILED` and the actual exception text — see the historical FAILED row in `pipeline_log` for a real example caught during development.


In [0]:
%run ./00_utils


In [0]:
import uuid
RUN_ID = str(uuid.uuid4())

In [0]:
from pyspark.sql import functions as F

SOURCE_PATH  = "/Volumes/workspace/taxi/raw/yellow_tripdata_2024-01 (2).parquet"
BRONZE_TABLE = "workspace.taxi.bronze_yellow_taxi"

try:
    df_raw = spark.read.parquet(SOURCE_PATH)
    rows_in = df_raw.count()

    df_bronze = (
        df_raw
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.lit(SOURCE_PATH))
    )

    (df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE))

    rows_out = spark.table(BRONZE_TABLE).count()

    log_pipeline_run(
        stage     = "bronze",
        rows_in   = rows_in,
        rows_out  = rows_out,
        status    = "SUCCESS"
    )

except Exception as e:
    log_pipeline_run(
        stage         = "bronze",
        rows_in       = -1,
        rows_out      = 0,
        status        = "FAILED",
        error_message = str(e)
    )
    raise   # re-raise so you still see the failure in the notebook output

/home/spark-32f0bb29-402a-4009-b3ce-a9/.ipykernel/2054/command-8792298986961328-153186287:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(PIPELINE_NAME, RUN_ID, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],


[bronze] SUCCESS | rows_in=2,964,624 rows_out=2,964,624


Loaded helpers: LOG_TABLE, LOG_SCHEMA, log_pipeline_run, PIPELINE_NAME


In [0]:
# Confirm the table exists, has lineage columns, and has version history
spark.sql(f"SELECT ingestion_timestamp, source_file, VendorID, tpep_pickup_datetime, fare_amount FROM {BRONZE_TABLE} LIMIT 5").show(truncate=False)
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").show(truncate=False)

+--------------------------+---------------------------------------------------------------+--------+--------------------+-----------+
|ingestion_timestamp       |source_file                                                    |VendorID|tpep_pickup_datetime|fare_amount|
+--------------------------+---------------------------------------------------------------+--------+--------------------+-----------+
|2026-05-05 10:08:09.090307|/Volumes/workspace/taxi/raw/yellow_tripdata_2024-01 (2).parquet|2       |2024-01-01 00:57:55 |17.7       |
|2026-05-05 10:08:09.090307|/Volumes/workspace/taxi/raw/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:03:00 |10.0       |
|2026-05-05 10:08:09.090307|/Volumes/workspace/taxi/raw/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:17:06 |23.3       |
|2026-05-05 10:08:09.090307|/Volumes/workspace/taxi/raw/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:36:38 |10.0       |
|2026-05-05 10:08:09.090307|/Volumes/workspace/taxi/raw

In [0]:
spark.sql("""
    SELECT pipeline_name, run_id, stage, rows_in, rows_out, status, error_message, run_timestamp
    FROM workspace.taxi.pipeline_log
    ORDER BY run_timestamp DESC
""").show(truncate=False)

+---------------+------------------------------------+------+-------+--------+-------+---------------------------------------------------------------------------+--------------------------+
|pipeline_name  |run_id                              |stage |rows_in|rows_out|status |error_message                                                              |run_timestamp             |
+---------------+------------------------------------+------+-------+--------+-------+---------------------------------------------------------------------------+--------------------------+
|nyc_taxi_yellow|07c8e520-8047-4816-b254-ae55bb3e6d9d|bronze|2964624|2964624 |SUCCESS|NULL                                                                       |2026-05-05 10:08:12.485299|
|nyc_taxi_yellow|01c11417-cd27-4f80-bc4d-5b758706eeb2|bronze|-1     |0       |FAILED |[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.|2026-05-05 10:07:09.445778|
+---------------+---------------------------------